# 05｜Panda 环境一次 step 的完整数据流

对应[第 05 章](../course/05-panda-environment-source.md)。本 Notebook 的小函数只是可执行示意图；正式环境语义以 [`pick_cartesian.py`](../../mujoco_playground/_src/manipulation/franka_emika_panda/pick_cartesian.py) 为准。

## 学习目标

能从策略动作追到动作历史、缩放/裁剪、IK、物理、奖励、终止和下一帧，并指出 guide-state 与 reward progress 的位置。

## 本节知识地图

| 知识点 | 一句话解释 | 掌握检查 |
| --- | --- | --- |
| reset | 创建物理状态、目标、观察和历史量 | 能列出随机量与固定量 |
| action path | 三维策略动作变为 y/z 增量与夹爪命令 | 能说出单位和 `action_scale` |
| IK + physics | IK 只给关节控制，接触运动仍由物理推进 | 不把 IK 有解等同成功 |
| reward progress | 当前总潜势只在超过历史最佳时给正增量 | 不把 raw terms 直接相加当 step reward |
| observation/done | 用新物理状态判断成功并生成下一帧 | 能说出严格顺序 |
| guide-state | 训练探索辅助，正式评估必须关闭 | 能定位完整性边界 |

## 关键概念与符号

策略动作 shape 为 `[3]`，语义是 y、z、gripper；环境内部扩成 `[x,y,z,gripper]`，其中 x 增量固定为 0。`action_scale=0.005` 表示单个控制步最大平移 5 mm。

## 开始前诊断

不看源码回答：① reward 在 physics 前还是后计算？② IK 有解是否保证抓取？③ 下一帧 RGB 应来自动作前还是动作后的状态？

## 先预测

给动作 `[1,-1,-1]`：预测三个分量语义、缩放后的 y/z 位移，以及哪个模块把笛卡尔目标变成关节控制。

## 运行与观察

先建立一条带命名中间量的教学 trace，再用真实源码验证顺序。教学 trace 不模拟完整接触动力学，也不产生性能证据。

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from course_feedback import check_choice, check_value, save_progress
from course_utils import assert_course_kernel

assert_course_kernel(ROOT)

## Worked example

下面把一次 step 拆成可见阶段。观察 `scaled_yz` 是米，`desired_tip_yz` 仍是笛卡尔目标；只有 IK 阶段才产生 actuator control。

In [ ]:
def didactic_step(action, *, tip_yz=(0.0, 0.08), box_yz=(0.01, 0.0), prev_best=0.0):
    action = np.asarray(action, dtype=float)
    scaled_yz = action[:2] * 0.005
    desired_tip_yz = np.clip(np.asarray(tip_yz) + scaled_yz, [-0.32, 0.02], [0.32, 0.50])
    ik_control = {'joint_target_available': True, 'gripper_closed': bool(action[2] < 0)}
    physics_tip_yz = desired_tip_yz.copy()  # 教学近似；正式环境由 MJX 接触动力学推进
    distance = float(np.linalg.norm(physics_tip_yz - np.asarray(box_yz)))
    total_potential = float(np.exp(-20.0 * distance))
    reward_progress = max(total_potential - prev_best, 0.0)
    success = abs(float(box_yz[1]) - 0.20) < 0.05
    return {
        'action': action, 'scaled_yz': scaled_yz, 'desired_tip_yz': desired_tip_yz,
        'ik_control': ik_control, 'physics_tip_yz': physics_tip_yz,
        'distance': distance, 'total_potential': total_potential,
        'reward_progress': reward_progress, 'success': success,
        'next_observation_shape': (64, 64, 3),
    }

trace = didactic_step([1.0, -1.0, -1.0])
for key, value in trace.items():
    print(f'{key:24s}: {value}')

### 用真实源码验证顺序

下面不执行昂贵环境，只读取源码并检查关键语句的先后位置。若上游实现变化导致顺序不同，断言会要求重新审计，而不是悄悄沿用旧图。

In [ ]:
SOURCE_PATH = ROOT / 'mujoco_playground/_src/manipulation/franka_emika_panda/pick_cartesian.py'
source_text = SOURCE_PATH.read_text(encoding='utf-8')
step_text = source_text[source_text.index('  def step('):source_text.index('  def _get_success(')]
ANCHORS = [
    'action_history =',
    'increment = jp.zeros(4)',
    'ctrl, new_tip_position, no_soln',
    'data = mjx_env.step',
    'raw_rewards = self._get_reward',
    'reward = jp.maximum',
    'obs = self._get_obs',
]
positions = [step_text.index(anchor) for anchor in ANCHORS]
for index, anchor in enumerate(ANCHORS, start=1):
    print(f'{index}. {anchor}')
assert positions == sorted(positions)

## 故意出错

错误实现先用旧 `data` 算 reward，再推进 physics。它会让奖励和下一 observation 描述不同时间点。根据真实源码顺序修正下面答案。

In [ ]:
reward_timing = 'before physics'  # 故意错误：改成 after physics
check_choice(
    'reward timing', reward_timing, 'after physics',
    hint='compare the mjx_env.step and _get_reward anchors above',
    explanation='reward, success, done, and next observation use the stepped data.',
)

## 动手修改

移动 y、z、gripper 控件，再运行 trace。先预测哪些字段改变；教学函数不会假装自己模拟了真实抓取。

In [ ]:
y_action = widgets.FloatSlider(value=0.5, min=-1, max=1, step=0.25, description='y action')
z_action = widgets.FloatSlider(value=-0.5, min=-1, max=1, step=0.25, description='z action')
gripper_action = widgets.ToggleButtons(options=[('open', 1.0), ('close', -1.0)], value=-1.0, description='gripper')
display(y_action, z_action, gripper_action)

In [ ]:
trial_trace = didactic_step([y_action.value, z_action.value, gripper_action.value])
print('scaled movement (m):', trial_trace['scaled_yz'])
print('desired tip yz:     ', trial_trace['desired_tip_yz'])
print('gripper closed:     ', trial_trace['ik_control']['gripper_closed'])
plt.figure(figsize=(5, 4))
plt.scatter([0.01], [0.0], s=100, label='cube')
plt.arrow(0.0, 0.08, *trial_trace['scaled_yz'], width=0.001, length_includes_head=True, color='tab:red')
plt.scatter(*trial_trace['desired_tip_yz'], label='desired tip')
plt.xlabel('world y'); plt.ylabel('world z'); plt.xlim(-0.03, 0.04); plt.ylim(-0.01, 0.11); plt.legend(); plt.show()

## 自测

把 shape、单位和源码顺序连成一条链。

In [ ]:
assert np.allclose(trace['scaled_yz'], [0.005, -0.005])
assert trace['ik_control']['gripper_closed']
assert trace['next_observation_shape'] == (64, 64, 3)
assert positions == sorted(positions)
assert ANCHORS.index('data = mjx_env.step') < ANCHORS.index('raw_rewards = self._get_reward')
print('PASS: action semantics, scale, source order, and next RGB shape')

## 项目源码连接

继续在真实文件中定位 `reset`、`step`、`_get_success`、`_move_tip`。尤其检查：guide-state 在 newly-reset 后发生；physics 在 reward 前；vision success 只比较高度；RGB 来自新 data。把行号写入[源码审计模板](../templates/source-audit.md)。

## Exit ticket

① 三维 action 的语义；② IK 与 physics 的职责边界；③ reward progress 与 raw reward 的区别；④ 正式评估如何处理 guide-state。

In [ ]:
exit_answers = {
    'action': 'y z gripper',
    'ik': 'joint control',
    'progress': 'positive improvement over best',
    'guide_eval': 'zero',
}
exit_ticket_passed = all([
    check_choice('action', exit_answers['action'], 'y z gripper', hint='x is fixed internally.', explanation='Correct three action semantics.'),
    check_choice('IK output', exit_answers['ik'], 'joint control', hint='contact still needs physics.', explanation='IK maps Cartesian target to joint control.'),
    check_choice('progress reward', exit_answers['progress'], 'positive improvement over best', hint='compare total with prev_reward.', explanation='Only new potential highs receive progress.'),
    check_choice('guide in evaluation', exit_answers['guide_eval'], 'zero', hint='training aid cannot enter formal reset.', explanation='Formal evaluation forces guide probability to zero.'),
])
SAVE_PROGRESS = False
if SAVE_PROGRESS:
    save_progress(ROOT, '05-dataflow', {'action': 'green', 'ik-physics': 'green', 'reward': 'green', 'guide': 'green'}, exit_ticket_passed=exit_ticket_passed)

## 学完请记住

1. 动作先经过历史、缩放和裁剪；2. IK 产生关节控制，物理产生接触后的新状态；3. reward/success/done/下一帧都读取新状态；4. progress 只奖励超过历史最佳的增量；5. guide-state 是训练辅助，正式评估为零。

## 反思与记录

在 `notes/05-panda-dataflow.md` 画出一次真实 step，并为每条箭头写 shape、单位、源码函数。教学 trace 通过不等于环境源码审计完成。